# Preprocessing — Pruning Anomaly Filter
Adding the Near_Pruning_Flag filter. Post-pruning recovery fields have Target_Days of 37-150 days which are biologically distinct and never seen in training data.

In [ ]:
import pandas as pd, numpy as np, warnings
warnings.filterwarnings('ignore')
from pathlib import Path
from sklearn.impute import SimpleImputer

BASE = Path(".")
df24 = pd.read_csv(BASE / "STEMS_Train_2024.csv")
df25 = pd.read_csv(BASE / "STEMS_Validate_2025.csv")
df26 = pd.read_csv(BASE / "STEMS_Test_2026.csv")

LEAK = ["Annual_Rounds","Months_In_Season","Year","Season","Division","Field_No","Target_Lag1","Target_Lag2"]
df_train = pd.concat([df24,df25], ignore_index=True)
df_test  = df26.copy()

for df in [df_train,df_test]:
    df.drop(columns=[c for c in LEAK if c in df.columns], inplace=True)

print(f"Before pruning filter — Train: {df_train.shape[0]}  Test: {df_test.shape[0]}")

# Apply pruning filter
df_train = df_train[df_train["Near_Pruning_Flag"]==0].reset_index(drop=True)
df_test  = df_test[df_test["Near_Pruning_Flag"]==0].reset_index(drop=True)
df_train.drop(columns=["Near_Pruning_Flag"], inplace=True)
df_test.drop(columns=["Near_Pruning_Flag"],  inplace=True)

print(f"After  pruning filter — Train: {df_train.shape[0]}  Test: {df_test.shape[0]}")
print(f"Removed from test: 9 post-pruning recovery fields")


In [ ]:
TARGET = "Target_Days"
num_cols = [c for c in df_train.select_dtypes(include=[np.number]).columns if c != TARGET]
imp = SimpleImputer(strategy='mean')
X_train = imp.fit_transform(df_train[num_cols])
X_test  = imp.transform(df_test[num_cols])
y_train = df_train[TARGET].values
y_test  = df_test[TARGET].values

print(f"X_train: {X_train.shape}  y_train: {y_train.shape}")
print(f"X_test : {X_test.shape}   y_test : {y_test.shape}")
print(f"y_train range: {y_train.min():.2f} — {y_train.max():.2f} days")
print(f"y_test  range: {y_test.min():.2f} — {y_test.max():.2f} days")
print("\nImprovements so far: leakage removed, pruning filter applied")
print("Still missing: feature engineering, outlier clipping, proper scaling")



- Test set: 71 → 62 clean rows
- Training set stays at 141 (no flagged rows in 2024/2025)
- Removes biologically abnormal fields that model has never seen patterns for

# Preprocessing for Feature Engineering
Adding 5 domain-driven engineered features: Soil_Index, Yield_Eff, Prune_Age_Ratio, Rain_Trend, Growth_per_Prod.

In [ ]:
import pandas as pd, numpy as np, warnings
warnings.filterwarnings('ignore')
from pathlib import Path
from sklearn.impute import SimpleImputer

BASE = Path(".")
df24 = pd.read_csv(BASE / "STEMS_Train_2024.csv")
df25 = pd.read_csv(BASE / "STEMS_Validate_2025.csv")
df26 = pd.read_csv(BASE / "STEMS_Test_2026.csv")

LEAK = ["Annual_Rounds","Months_In_Season","Year","Season","Division","Field_No","Target_Lag1","Target_Lag2"]
df_train = pd.concat([df24,df25],ignore_index=True)
df_test  = df26.copy()
for df in [df_train,df_test]: df.drop(columns=[c for c in LEAK if c in df.columns],inplace=True)
df_train = df_train[df_train["Near_Pruning_Flag"]==0].reset_index(drop=True)
df_test  = df_test[df_test["Near_Pruning_Flag"]==0].reset_index(drop=True)
for df in [df_train,df_test]: df.drop(columns=["Near_Pruning_Flag"],inplace=True)


In [ ]:
def engineer(df):
    df = df.copy()
    # Soil quality composite
    df["Soil_Index"] = df["Soil_Carbon"] / (df["Soil_pH"].replace(0, np.nan) + 1e-9)
    # Yield efficiency per hectare
    df["Yield_Eff"] = df["Yield_Prev_Year"] / (df["Extent_Hect"].replace(0, np.nan) + 1e-9)
    # Prune stage relative to plant age
    df["Prune_Age_Ratio"] = df["Prune_Cycle_Stage"] / (df["Age_Months"] / 12 + 1e-9)
    # Rainfall trend: is rainfall increasing or decreasing?
    if "Rainfall_Lag1" in df.columns and "Rainfall_Lag3" in df.columns:
        df["Rain_Trend"] = df["Rainfall_Lag1"] - df["Rainfall_Lag3"]
    # Growth efficiency
    if "Growth_Response" in df.columns and "Field_Productivity" in df.columns:
        df["Growth_per_Prod"] = df["Growth_Response"] / (df["Field_Productivity"] + 1e-9)
    return df

df_train = engineer(df_train)
df_test  = engineer(df_test)

new_features = ["Soil_Index","Yield_Eff","Prune_Age_Ratio","Rain_Trend","Growth_per_Prod"]
print("new features")
for f in new_features:
    col = df_train[f].replace([np.inf,-np.inf],np.nan)
    print(f"  {f:<22} range=[{col.min():.3f}, {col.max():.3f}]  NaN={col.isna().sum()}")


In [ ]:
TARGET = "Target_Days"
num_cols = [c for c in df_train.select_dtypes(include=[np.number]).columns if c != TARGET]
imp = SimpleImputer(strategy='mean')
X_train = imp.fit_transform(df_train[num_cols])
X_test  = imp.transform(df_test[num_cols])
y_train = df_train[TARGET].values; y_test = df_test[TARGET].values

print(f"Feature count before engineering: ~31")
print(f"Feature count after  engineering: {len(num_cols)} (+5 new features)")
print(f"X_train: {X_train.shape}")
print(f"NaN after imputation: {np.isnan(X_train).sum()}")
print("\nStill missing: outlier clipping, proper MinMaxScaler, KNN imputation")


##  Feature Engineering Added
- 5 new domain-driven features: Soil_Index, Yield_Eff, Prune_Age_Ratio, Rain_Trend, Growth_per_Prod
- All use only lag values — no future data leakage
- NaN values from division-by-zero handled with ε offset
- Total features: ~36

# Outlier Clipping
Adding 1st-99th percentile clipping on training data. Growth_Response reaches 9.7 million and extreme values distort distance-based models like SVR.

In [ ]:
import pandas as pd, numpy as np, warnings
warnings.filterwarnings('ignore')
from pathlib import Path
from sklearn.impute import SimpleImputer

BASE = Path(".")
df24 = pd.read_csv(BASE / "STEMS_Train_2024.csv")
df25 = pd.read_csv(BASE / "STEMS_Validate_2025.csv")
df26 = pd.read_csv(BASE / "STEMS_Test_2026.csv")

LEAK = ["Annual_Rounds","Months_In_Season","Year","Season","Division","Field_No","Target_Lag1","Target_Lag2"]
df_train = pd.concat([df24,df25],ignore_index=True)
df_test  = df26.copy()
for df in [df_train,df_test]: df.drop(columns=[c for c in LEAK if c in df.columns],inplace=True)
df_train=df_train[df_train["Near_Pruning_Flag"]==0].reset_index(drop=True)
df_test =df_test[df_test["Near_Pruning_Flag"]==0].reset_index(drop=True)
for df in [df_train,df_test]: df.drop(columns=["Near_Pruning_Flag"],inplace=True)

def engineer(df):
    df=df.copy()
    df["Soil_Index"]=df["Soil_Carbon"]/(df["Soil_pH"].replace(0,np.nan)+1e-9)
    df["Yield_Eff"]=df["Yield_Prev_Year"]/(df["Extent_Hect"].replace(0,np.nan)+1e-9)
    df["Prune_Age_Ratio"]=df["Prune_Cycle_Stage"]/(df["Age_Months"]/12+1e-9)
    df["Rain_Trend"]=df["Rainfall_Lag1"]-df["Rainfall_Lag3"]
    df["Growth_per_Prod"]=df["Growth_Response"]/(df["Field_Productivity"]+1e-9)
    return df

df_train=engineer(df_train); df_test=engineer(df_test)
TARGET="Target_Days"
num_cols=[c for c in df_train.select_dtypes(include=[np.number]).columns if c!=TARGET]
X_tr_raw=df_train[num_cols].copy(); X_te_raw=df_test[num_cols].copy()

print("Raw features range")
extremes = ["Growth_Response","Yield_Eff","Growth_per_Prod","Field_Productivity","Days_Since_Last_Pruning"]
for c in extremes:
    if c in X_tr_raw.columns:
        print(f"  {c:<28} min={X_tr_raw[c].min():.1f}  max={X_tr_raw[c].max():.1f}")


In [ ]:
# Apply clipping — train bounds only, applied to both
lo = X_tr_raw.quantile(0.01)
hi = X_tr_raw.quantile(0.99)
X_tr_clip = X_tr_raw.clip(lo, hi, axis=1)
X_te_clip = X_te_raw.clip(lo, hi, axis=1)

print("after clipping")
for c in extremes:
    if c in X_tr_clip.columns:
        rows_clipped = (X_tr_raw[c] != X_tr_clip[c]).sum()
        print(f"  {c:<28} max={X_tr_clip[c].max():.1f}  rows_clipped={rows_clipped}")

print("\nTest set clipped using train bounds (corredect — no data leakage)")


In [ ]:
imp=SimpleImputer(strategy='mean')
X_train=imp.fit_transform(X_tr_clip); X_test=imp.transform(X_te_clip)
y_train=df_train[TARGET].values; y_test=df_test[TARGET].values

print(f"X_train: {X_train.shape}  NaN: {np.isnan(X_train).sum()}")
print(f"X_test : {X_test.shape}   NaN: {np.isnan(X_test).sum()}")
print("\n leakage removed, pruning filter, feature engineering, clipping")



## Outlier Clipping Added
- Growth_Response was reaching 9.7M — now compressed to 99th percentile of training data
- Clipping bounds computed from training data only and applied to test set
- Prevents extreme values from dominating distance calculations in SVR